# **Обучение CatBoost модели и её применение**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 26-03-2026  

**Цель:** Обучение CatBoost модели, примение модели на тестовой выборке для получения результатов в kaggle соревновании ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview), а также сбор всех необходимых для дашборда метрик

#### Необходимые библиотеки и настройка графиков:

In [1]:
%pip install catboost -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import gzip
import zipfile
from collections import Counter
from datetime import datetime

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score

In [3]:
%config InlineBackend.figure_format = 'retina'

sns.set(style='darkgrid', palette='deep')

plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'

### 0. Загрука набора данных c kaggle
Скачиваем все файлы с соревнования ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) для дальнейшего использования.

In [4]:
# os.environ['KAGGLE_API_TOKEN'] = "ВАШ_KAGGLE_API_TOKEN"
# os.environ['KAGGLE_API_TOKEN'] = "KGAT_4557a5e2791ef4a201a0119ee0cea405"

print("✅ Kaggle API ключ установлен!")

✅ Kaggle API ключ установлен!


In [5]:
%pip install kaggle -q
!kaggle competitions download -c avazu-ctr-prediction

Note: you may need to restart the kernel to use updated packages.
  2%|▉                                     | 30.0M/1.19G [00:03<02:01, 10.3MB/s]^C


In [6]:
with zipfile.ZipFile('../avazu-ctr-prediction.zip', 'r') as zip_ref:
    zip_ref.extractall('../avazu-ctr-prediction')

### 1. Обучение CatBoost модели

При обучении будем использовать Out-Of-Time Validation, это лучше отражает качество моделей на времязависимых данных. Нам повезло и набор данных хранится уже осторированным. Как мы узнали ранее, в датасете $40 428 967$ строк. Оставим первые $30$ млн. из них на обучение, остальные олтложим для валидации. Получим, что примерно $74$% от тренировочного набора данных уйдет на обучающую выборку и $26$% на валидацонную.

Обучаться будем на GPU, а именно с помощью Colab и его бесплатного T4. Поэтому придется подключить Google Drive для возможности вытащить потом файлы, созданные во время ноутбука.

Все признаки в датасете категориальные, кроме `hour`. Но этот признак нельзя передавать как численный: так мы неявно зададим порядок по времени. Я предлагаю вытащить из колонки 2 категориальных признака: номер часа в сутках и номер дня недели. Если оставить 2 признака числовыми, то мы опять сталкнемся с проблеммой неявного попрядка на данных.

#### Подключим Google Drive

In [7]:
# from google.colab import drive

# drive.mount('/content/drive')

In [8]:
# folder_path = '/content/drive/MyDrive/colab_course_work/catboost'
folder_path = '../catboost'

#### Подготовим данные перед обучением модели

In [9]:
def data_tranformer(df: pd.DataFrame):
    dt = pd.to_datetime(df['hour'], format='%y%m%d%H')
    df['day_of_week'] = dt.dt.dayofweek
    df['hour_of_day'] = dt.dt.hour

    return df.drop(columns=['id', 'hour'], axis=1).astype(str)

In [11]:
import pandas as pd
import gc
import os

os.makedirs(f'{folder_path}/preprocessed_data', exist_ok=True)


print("⏳ Начинаем препроцессинг данных и разбивку на Train и Val...")

chunk_size = 5_000_000
# train_file = 'avazu-ctr-prediction/train.gz'
train_file = '../avazu-ctr-prediction/train.gz'
chunk_iterator = pd.read_csv(train_file, compression='gzip', chunksize=chunk_size)

for chunk_num, chunk in enumerate(chunk_iterator, 1):
    processed_chunk = data_tranformer(chunk)

    if chunk_num <= 6:
        processed_chunk.to_csv(
            f'{folder_path}/preprocessed_data/train_processed.csv',
            mode='w' if chunk_num == 1 else 'a',
            header=(chunk_num == 1),
            index=False
        )
        print(f"📦 Чанк №{chunk_num} обработан и добавлен в train_processed.csv")
    else:
        processed_chunk.to_csv(
            f'{folder_path}/preprocessed_data/val_processed.csv',
            mode='w' if chunk_num == 7 else 'a',
            header=(chunk_num == 7),
            index=False
        )
        print(f"🧪 Чанк №{chunk_num} обработан и добавлен в val_processed.csv")

    del chunk, processed_chunk
    gc.collect()

print("✅ Препроцессинг завершен! Данные готовы к стримингу.")

⏳ Начинаем препроцессинг данных и разбивку на Train и Val...
📦 Чанк №1 обработан и добавлен в train_processed.csv
📦 Чанк №2 обработан и добавлен в train_processed.csv
📦 Чанк №3 обработан и добавлен в train_processed.csv
📦 Чанк №4 обработан и добавлен в train_processed.csv
📦 Чанк №5 обработан и добавлен в train_processed.csv
📦 Чанк №6 обработан и добавлен в train_processed.csv
🧪 Чанк №7 обработан и добавлен в val_processed.csv
🧪 Чанк №8 обработан и добавлен в val_processed.csv
🧪 Чанк №9 обработан и добавлен в val_processed.csv
✅ Препроцессинг завершен! Данные готовы к стримингу.


In [12]:
print("⏳ Создаем файл cd.txt...")

sample_df = pd.read_csv(f'{folder_path}/preprocessed_data/train_processed.csv', nrows=0)
columns = sample_df.columns
target = 'click'

with open(f'{folder_path}/preprocessed_data/cd.txt', 'w') as f:
    for i, col in enumerate(columns):
        if col == target:
            f.write(f"{i}\tLabel\n")
        else:
            f.write(f"{i}\tCateg\n")

print("✅ Файл cd.txt успешно создан!")

⏳ Создаем файл cd.txt...
✅ Файл cd.txt успешно создан!


#### Перейдем к обучению

In [13]:
print("⚙️ Инициализация Pool-объектов CatBoost'а...")

train_pool = Pool(
    data=f'{folder_path}/preprocessed_data/train_processed.csv',
    column_description=f'{folder_path}/preprocessed_data/cd.txt',
    has_header=True,
    delimiter=','
)

val_pool = Pool(
    data=f'{folder_path}/preprocessed_data/val_processed.csv',
    column_description=f'{folder_path}/preprocessed_data/cd.txt',
    has_header=True,
    delimiter=','
)

⚙️ Инициализация Pool-объектов CatBoost'а...


In [15]:
model_params = {
    'iterations': 3000,
    'learning_rate': 0.03,
    'depth': 9,
    'l2_leaf_reg': 3,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'od_type': 'Iter',
    'od_wait': 200,
    'use_best_model': True,
    'max_ctr_complexity': 3,
    'random_strength': 1,
    'has_time': True,
    'bagging_temperature': 0.5,
    'border_count': 128,
    'random_seed': 67,
    'task_type': 'CPU',
    'thread_count': -1, # Идем на взлет
    'verbose': 100
}

model = CatBoostClassifier(**model_params)

print("🚀 Начинаем обучение...")
model.fit(train_pool, eval_set=val_pool)

os.makedirs(f'{folder_path}/models', exist_ok=True)
model_path = f'{folder_path}/models/catboost_ctr_model.cbm'
model.save_model(model_path)

print(f"\n🎉 Обучение завершено! Модель сохранена в {model_path}")

🚀 Начинаем обучение...
0:	test: 0.7314506	best: 0.7314506 (0)	total: 2m 3s	remaining: 4d 7h 4m 58s


KeyboardInterrupt: 

Обучение прошло успешно, теперь посмотрим на получившиеся на валидационной выборке метрики:

In [ ]:
val_y_true = val_pool.get_label()

val_y_pred_proba = model.predict_proba(val_pool)[:, 1]
val_y_pred_class = model.predict(val_pool)

In [ ]:
roc_auc = roc_auc_score(val_y_true, val_y_pred_proba)
logloss = log_loss(val_y_true, val_y_pred_proba)
acc = accuracy_score(val_y_true, val_y_pred_class)

print(f"📊 Итоговые метрики (OOT выборка на {f'{val_size:_}'.replace('_', ' ')} строк):")

metrics_df = pd.DataFrame({
    'Метрика': ['ROC-AUC Score', 'Log Loss', 'Accuracy'],
    'Значение': [roc_auc, logloss, acc]
})
metrics_df['Значение'] = metrics_df['Значение'].round(4)

metrics_df

📊 Итоговые метрики (OOT выборка на 10 428 967 строк):


,Метрика,Значение
0,ROC-AUC Score,0.7074
1,Log Loss,0.4196
2,Accuracy,0.8295


### 2. Применение модели на тестовой выборке и отправка решения на kaggle

Считаем тестовую выборку:

In [ ]:
warnings.filterwarnings('ignore')

test_file = "avazu-ctr-prediction/test.gz"

print("⏳ Читаем test.gz...")
test_df = pd.read_csv(test_file, compression='gzip', dtype={'id': str})

print(f"✅ Итого загружено: {f"{len(test_df):_}".replace('_', ' ')} строк")

ram_usage = test_df.memory_usage(deep=True).sum() / 1024**3
print(f"📊 Объем памяти DataFrame: {ram_usage:.2f} GB")

⏳ Читаем test.gz...
✅ Итого загружено: 4 577 464 строк
📊 Объем памяти DataFrame: 2.92 GB


А также модель:

In [ ]:
print("⏳ Достаем обученную модель...")
model = CatBoostClassifier()
model.load_model(model_path)

print(f"Количество признаков в модели: {len(model.feature_names_)}")

⏳ Достаем обученную модель...
Количество признаков в модели: 23


Применим модель к тестовому набору данных:

In [ ]:
ids = test_df['id']
X_test = data_tranformer(test_df)

del test_df
gc.collect()

print("\n⏳ Генерация предсказаний...")
y_pred_proba = model.predict_proba(X_test)[:, 1]

del X_test
gc.collect()


⏳ Генерация предсказаний...


0

Выведем минимальную статистику по полученным предсказаниям:

In [ ]:
stats_data = [
    ('Средняя вероятность', y_pred_proba.mean()),
    ('Медианная вероятность', np.median(y_pred_proba)),
    ('Стандартное отклонение', y_pred_proba.std()),
    ('Минимум', y_pred_proba.min()),
    ('Максимум', y_pred_proba.max())
]
stats_df = pd.DataFrame(stats_data, columns=['Статистика', 'Значение'])
stats_df['Значение'] = stats_df['Значение'].round(4)

print("📊 Статистики предсказаний на тестовой выборке:")
display(stats_df)


quantiles = []
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    quantiles.append((f'{int(q*100)}%', np.quantile(y_pred_proba, q)))
quantiles_df = pd.DataFrame(quantiles, columns=['Квантиль', 'Значение'])

print("📊 Квантили на тестовой выборке:")
display(quantiles_df)


📊 Статистики предсказаний на тестовой выборке:


,Статистика,Значение
0,Средняя вероятность,0.1995
1,Медианная вероятность,0.1946
2,Стандартное отклонение,0.1486
3,Минимум,0.0001
4,Максимум,0.9979


📊 Квантили на тестовой выборке:


,Квантиль,Значение
0,10%,0.027373
1,25%,0.077421
2,50%,0.194555
3,75%,0.279083
4,90%,0.343303


*Наконец, сохраненим результат и сделаем посылку submission'а на kaggle:*

In [ ]:
submission_file = f'{folder_path}/data/submission.csv'
os.makedirs(f'{folder_path}/data', exist_ok=True)

In [ ]:
submission = pd.DataFrame({'id': ids, 'click': y_pred_proba})
submission.to_csv(submission_file, index=False)
submission.to_csv('submission.csv', index=False)
print(f"✅ Результаты успешно сохранены в: {submission_file}")

✅ Результаты успешно сохранены в: data/submission.csv


In [ ]:
!kaggle competitions submit -c avazu-ctr-prediction -f submission.csv  -m "Best CatBoost submission"

100%|████████████████████████████████████████| 175M/175M [00:28<00:00, 6.42MB/s]
Successfully submitted to Click-Through Rate Prediction